In [7]:
import sys
import os

# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src")))

In [30]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema
import pandas as pd

from pyspark.sql.functions import udtf
from pyspark.sql.types import StructType, StructField, StringType

In [31]:

@pandas_udf(get_feature_schema(), PandasUDFType.GROUPED_MAP)
def extract_features_udtf(pdf):
    rows = []
    for _, row in pdf.iterrows():
        subject_id = row["SubjectID"]
        try:
            epochs = processSub(subject_id, derivatives=False)
            for i, epoch in enumerate(epochs):
                epoch_id = f"{subject_id}_ep{i}"
                features = processEpoch(epoch)
                for (electrode, band), stats in features:
                    # Assuming `stats` is a tuple with (mean, variance, skewness, kurtosis)
                    rows.append((subject_id, epoch_id, band, electrode, *stats))
        except Exception as e:
            print(f"Error processing {subject_id}: {e}")
    return pd.DataFrame(rows, columns=[f.name for f in get_feature_schema()])

In [49]:
@pandas_udf(get_subject_schema(), PandasUDFType.GROUPED_MAP)
def generate_subjects_udtf(pdf):
    # Read the participants.tsv file
    participantsInfo = pd.read_table('../ds004504/participants.tsv')
    
    # Filter the subjects by group and prepare the data
    subjects = []
    for group, group_name in [("A", "GroupA"), ("C", "GroupC"), ("F", "GroupD")]:
        group_subjects = participantsInfo[participantsInfo["Group"] == group]["participant_id"].tolist()
        subjects.extend([{"SubjectID": sub, "Group": group_name} for sub in group_subjects])
    
    # Return a DataFrame with the subjects and their groups
    return pd.DataFrame(subjects)


In [44]:
from pyspark.sql.functions import udtf
from pyspark.sql.types import StructType, StructField, StringType

# @udtf(returnType=StructType([
#     StructField("SubjectID", StringType(), False),
#     StructField("Group", StringType(), False)
# ]))
class GenerateSubjectsUDTF:
    def eval(self):
        participantsInfo = pd.read_table('../ds004504/participants.tsv')
        for group, group_name in [("A", "GroupA"), ("C", "GroupC"), ("F", "GroupD")]:
            group_subjects = participantsInfo[participantsInfo["Group"] == group]["participant_id"].tolist()
            for sub in group_subjects:
                yield sub, group_name

In [45]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("YourAppName") \
    .getOrCreate()

In [46]:
# Assuming you have created the Spark session as shown above

# Instantiate your UDTF
# generate_subjects_udtf = udtf(GenerateSubjectsUDTF)
generate_subjects_udtf = udtf(GenerateSubjectsUDTF, returnType=get_subject_schema())



In [47]:
# Create a dummy DataFrame to trigger the UDTF
df = spark.range(1)


In [48]:
print(df.head())

Row(id=0)


In [ ]:

# Apply the UDTF
result = df.select(generate_subjects_udtf().alias("subjects"))

# Show the results
result.show()

# Attemtping to use the udtf framework form pyspark, but its difficult  https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udtf.html?highlight=udtf

In [33]:
@udtf(returnType=get_subject_schema())
class GenerateSubjectsUDTF:
    def eval(self):
        participantsInfo = pd.read_table('../ds004504/participants.tsv')
        for group, group_name in [("A", "GroupA"), ("C", "GroupC"), ("F", "GroupD")]:
            group_subjects = participantsInfo[participantsInfo["Group"] == group]["participant_id"].tolist()
            for sub in group_subjects:
                yield sub, group_name

In [34]:
# Usage
generate_subjects_udtf = udtf(GenerateSubjectsUDTF)
result = spark.range(1).select(generate_subjects_udtf().alias("subjects"))

TypeError: udtf() missing 1 required keyword-only argument: 'returnType'

# End of tyring to use udtf

# Initialize Spark Session
spark = SparkSession.builder.appName("SubjectPopulation").getOrCreate()

# Create a dummy DataFrame to trigger the UDTF
subjects_df = spark.createDataFrame([("dummy",)], ["key"])

# Apply the UDTF
subject_df = subjects_df.groupby("key").apply(generate_subjects_udtf)

# Display or save the results
subject_df.show(n=subject_df.count(), truncate=False)# Optionally, you can save this DataFrame
# subject_df.write.parquet("path/to/save/subjects")

In [61]:
print(subject_df.groupBy('Group').count())

DataFrame[Group: string, count: bigint]


In [62]:
group_counts = subject_df.groupBy("Group").count()
group_counts.show()

+------+-----+
| Group|count|
+------+-----+
|GroupA|   36|
|GroupD|   23|
|GroupC|   29|
+------+-----+



In [79]:
group_counts = subject_df.groupBy("SubjectID").count()
group_counts.show()

[Stage 93:>                                                         (0 + 1) / 1]

+---------+-----+
|SubjectID|count|
+---------+-----+
|  sub-067|    1|
|  sub-058|    1|
|  sub-081|    1|
|  sub-057|    1|
|  sub-020|    1|
|  sub-033|    1|
|  sub-037|    1|
|  sub-055|    1|
|  sub-077|    1|
|  sub-083|    1|
|  sub-012|    1|
|  sub-002|    1|
|  sub-062|    1|
|  sub-011|    1|
|  sub-060|    1|
|  sub-069|    1|
|  sub-016|    1|
|  sub-082|    1|
|  sub-049|    1|
|  sub-073|    1|
+---------+-----+
only showing top 20 rows



In [70]:
# subject_df.show()
subject_df.show(1)

[Stage 81:>                                                         (0 + 1) / 1]

+---------+------+
|SubjectID| Group|
+---------+------+
|  sub-001|GroupA|
+---------+------+
only showing top 1 row



In [75]:
# Assuming generate_subjects_udtf is defined elsewhere
# subjects_df = generate_subjects_udtf(spark)

# Print the schema to check column names
subjects_df.printSchema()

root
 |-- key: string (nullable = true)



In [80]:
one_subject_df = subjects_df.select("`key`")

In [81]:
subjects_df.select("`key`").distinct().show()

+-----+
|  key|
+-----+
|dummy|
+-----+



In [ ]:
spark.stop()